# 🩺 Breast Cancer Detection using Machine Learning
### A Comparative Study of Classification Algorithms on the Wisconsin Diagnostic Breast Cancer (WDBC) Dataset

**Team:** NeuralNagpur (Palak Vijaywar)
**Internship:** InternPe — AI/ML Track
**Institution:** Priyadarshini College of Engineering (PCE), Nagpur — B.Tech AI & DS, 3rd Year

---

## Abstract
Breast cancer remains one of the most commonly diagnosed cancers among women worldwide, and early, accurate diagnosis significantly improves survival outcomes. This notebook presents a comparative machine learning study for classifying breast tumours as **malignant** or **benign** using the Wisconsin Diagnostic Breast Cancer (WDBC) dataset. Five classification algorithms — Logistic Regression, K-Nearest Neighbours, Support Vector Machine, Random Forest, and Naive Bayes — are trained, tuned, and evaluated using accuracy, precision, recall, F1-score, and ROC-AUC. The best-performing model is serialised for reuse in a lightweight prediction interface.


## 1. Introduction

Diagnostic decisions for breast cancer are traditionally based on fine needle aspirate (FNA) cytology, where cell nuclei characteristics (radius, texture, perimeter, area, smoothness, etc.) are measured. Machine learning offers a way to model the relationship between these quantitative features and diagnosis, acting as a **decision-support tool** for radiologists and pathologists — not a replacement for clinical judgement.

**Objectives:**
1. Perform exploratory data analysis (EDA) on the WDBC dataset.
2. Preprocess and scale features for model readiness.
3. Train and compare 5 classical ML algorithms.
4. Evaluate using multiple metrics (not just accuracy — critical in medical diagnosis where false negatives are costly).
5. Interpret feature importance to understand which cell characteristics matter most.
6. Package the best model for reuse/deployment.


In [ ]:
# 2. Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve,
                              confusion_matrix, classification_report)
import joblib

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)
RANDOM_STATE = 42


## 2. Dataset & Exploratory Data Analysis (EDA)

We use the **Wisconsin Diagnostic Breast Cancer dataset**, built into scikit-learn (originally from UCI ML Repository). It contains **569 samples** and **30 real-valued features** computed from digitised images of FNA biopsies, with a binary target: `0 = malignant`, `1 = benign`.


In [ ]:
# Load dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['diagnosis'] = data.target  # 0 = malignant, 1 = benign

print("Shape:", df.shape)
print("\nClass distribution:")
print(df['diagnosis'].value_counts().rename({0: 'Malignant', 1: 'Benign'}))
df.head()


In [ ]:
# Class balance visualisation
plt.figure(figsize=(5,4))
diag_labels = df['diagnosis'].map({0: 'Malignant', 1: 'Benign'})
sns.countplot(x=diag_labels, hue=diag_labels, palette=['#d62828', '#2a9d8f'], legend=False)
plt.title('Diagnosis Class Distribution')
plt.xlabel('')
plt.ylabel('Count')
plt.show()

df.describe().T.head(10)


In [ ]:
# Correlation heatmap (top features vs target)
corr = df.corr()['diagnosis'].abs().sort_values(ascending=False)[1:11]
plt.figure(figsize=(8,6))
sns.heatmap(df[corr.index.tolist() + ['diagnosis']].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Top 10 Features Correlated with Diagnosis')
plt.show()


## 3. Preprocessing

- Split into train (80%) / test (20%) sets, stratified on the target to preserve class balance.
- Standardise features using `StandardScaler` — essential for distance/margin-based models like KNN and SVM.


In [ ]:
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape, "| Test shape:", X_test_scaled.shape)


## 4. Model Building & Methodology

Five algorithms are trained on identical train/test splits for a fair comparison:

| Model | Why it's included |
|---|---|
| Logistic Regression | Strong, interpretable baseline for binary classification |
| K-Nearest Neighbours | Simple instance-based learner, sensitive to feature scaling |
| Support Vector Machine (RBF) | Effective in high-dimensional feature spaces |
| Random Forest | Ensemble method, handles non-linearity, gives feature importance |
| Gaussian Naive Bayes | Fast probabilistic baseline, assumes feature independence |

Each model is evaluated using **Accuracy, Precision, Recall, F1-score, and ROC-AUC**. In a medical context, **Recall (Sensitivity)** for the malignant class is especially important — missing a malignant case (false negative) is far costlier than a false alarm.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    'Naive Bayes': GaussianNB()
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False).reset_index(drop=True)
results_df


In [ ]:
# Visual comparison of models
results_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].plot(
    kind='bar', figsize=(10,6)
)
plt.title('Model Comparison Across Metrics')
plt.ylabel('Score')
plt.ylim(0.8, 1.02)
plt.xticks(rotation=20)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 5. Detailed Evaluation of the Best Model

Based on the comparison table above, we select the top-performing model (by F1-score) for deeper evaluation: confusion matrix, classification report, and ROC curve.


In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
print(f"Best model: {best_model_name}")

y_pred_best = best_model.predict(X_test_scaled)
y_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'])
plt.title(f'Confusion Matrix — {best_model_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print(classification_report(y_test, y_pred_best, target_names=['Malignant', 'Benign']))


In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba_best)
auc_score = roc_auc_score(y_test, y_proba_best)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'{best_model_name} (AUC = {auc_score:.3f})', color='#2a9d8f', linewidth=2)
plt.plot([0,1], [0,1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()


## 6. Feature Importance (Interpretability)

Even when Random Forest isn't the single best model, it's useful to inspect which cell-nuclei measurements drive predictions the most — this gives a clinically interpretable view of the model's reasoning.


In [ ]:
rf = models['Random Forest']
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8,6))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, palette='viridis', legend=False)
plt.title('Top 10 Most Important Features (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 7. Saving the Model for Reuse

We persist the best model and the scaler using `joblib`, so they can be loaded later in a script, API, or simple web demo without retraining.


In [ ]:
joblib.dump(best_model, 'breast_cancer_best_model.pkl')
joblib.dump(scaler, 'breast_cancer_scaler.pkl')
print("Saved: breast_cancer_best_model.pkl, breast_cancer_scaler.pkl")


## 8. Demo: Predicting on a New Sample

A quick sanity-check function simulating how this would be used in a real application — pass in the 30 measured features for a new patient sample and get a diagnosis + confidence score.


In [ ]:
def predict_diagnosis(sample_features, model=best_model, scaler=scaler):
    """
    sample_features: array-like of 30 values in the same order as `data.feature_names`
    Returns: (diagnosis label, confidence)
    """
    sample_df = pd.DataFrame([sample_features], columns=X.columns)
    sample_scaled = scaler.transform(sample_df)
    pred = model.predict(sample_scaled)[0]
    proba = model.predict_proba(sample_scaled)[0][pred]
    label = 'Benign' if pred == 1 else 'Malignant'
    return label, round(proba * 100, 2)

# Example: use an actual test-set row to demonstrate
sample = X_test.iloc[0].values
true_label = 'Benign' if y_test.iloc[0] == 1 else 'Malignant'
pred_label, confidence = predict_diagnosis(sample)

print(f"True label:      {true_label}")
print(f"Predicted label: {pred_label}  (confidence: {confidence}%)")


## 9. Conclusion & Future Work

This study compared five classical ML algorithms for breast cancer diagnosis on the WDBC dataset, achieving strong performance (typically 95–99% accuracy and ROC-AUC across models) with Logistic Regression, SVM, and Random Forest performing particularly well on this dataset.

**Limitations:**
- Dataset size (569 samples) is small relative to real-world clinical deployment needs.
- Model was not validated on external/hospital data — a necessary step before any clinical use.
- This is a research/educational prototype and **not a certified diagnostic tool**.

**Future Work:**
- Hyperparameter tuning via `GridSearchCV` / `Optuna` for further gains.
- Ensemble/stacking of top models.
- SHAP-based explainability for per-patient interpretation.
- Deployment as a Flask/Streamlit demo app with a simple UI for feature input.

---

## References
1. Wolberg, W.H., Street, W.N., Mangasarian, O.L. — *Breast Cancer Wisconsin (Diagnostic) Data Set*, UCI Machine Learning Repository.
2. Scikit-learn documentation — https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html

---
*Prepared by Palak Vijaywar for the InternPe AI/ML Internship. Collaboration and feedback welcome — feel free to fork, raise issues, or suggest improvements on GitHub.*
